# Data Engineering — Moduł 8: Podstawy streamingu i Apache Kafka

Moduły 2-7 dotyczyły świata **batch**: Airflow i dbt uruchamiają się w ustalonych
odstępach czasu i przetwarzają dane w paczkach. Ten moduł otwiera drugi świat --
**streaming**: przetwarzanie KAŻDEGO zdarzenia z osobna, w miarę jak napływa.
**Apache Kafka** to najpopularniejszy **message broker** (pośrednik komunikatów) do tego
celu -- fundament, na którym zbudujemy Spark Structured Streaming w Module 9.

## Spis treści
1. [Dlaczego Kafka? Problem rozprzężenia (decoupling)](#sec1)
2. [Podstawowe pojęcia: Topic, Partition, Producer, Consumer](#sec2)
3. [Consumer Groups i offsety](#sec3)
4. [Gwarancje dostarczenia](#sec4)
5. [Kod producenta i konsumenta](#sec5)
6. [Kafka Connect i Schema Registry](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


> ⚠️ **Ten moduł jest koncepcyjny -- bez lokalnego brokera Kafki**
>
> Uruchomienie prawdziwego brokera Kafki wymaga demona Dockera (niedostępnego w tym środowisku sandboksowym) albo pobrania binarek Kafki z serwerów Apache (zablokowane przez politykę sieciową tego środowiska -- dozwolone są tylko rejestry pakietów jak pip/npm). To dokładnie ta sama sytuacja co Delta Lake (Moduł 10) w kursie Databricks/PySpark: kod w tym module jest poprawny i gotowy do uruchomienia na prawdziwym brokerze (lokalny Docker, Confluent Cloud, albo Kafka wbudowana w Databricks), ale NIE wykonujemy go tutaj -- pokazujemy fragmenty kodu do przeczytania i wypróbowania we własnym środowisku. Sekcje 1-4 i 6 są w pełni konceptualne (bez kodu do uruchomienia); Moduł 9 pokaże, jak przetwarzać strumień danych NAPRAWDĘ, korzystając z wbudowanego w Sparka źródła zastępczego.

<a id="sec1"></a>
## 1. Dlaczego Kafka? Problem rozprzężenia (decoupling)

Wyobraź sobie system BEZ Kafki: aplikacja sklepu internetowego, gdy klient złoży
zamówienie, musi POWIADOMIĆ wszystkie zainteresowane systemy -- magazyn (żeby zarezerwować
towar), system płatności, system rekomendacji, system analityczny, system antyfraudowy.
Jeśli aplikacja woła każdy z tych systemów BEZPOŚREDNIO (np. przez API), powstaje kilka
problemów:

- Aplikacja musi ZNAĆ wszystkie systemy, które kiedykolwiek będą chciały wiedzieć o
  zamówieniu -- dodanie NOWEGO konsumenta (np. nowego systemu analitycznego) wymaga
  zmiany kodu aplikacji sklepu.
- Jeśli JEDEN z systemów jest wolny albo padnie, CAŁE zamówienie może się nie powieść.
- Trudno odtworzyć historię zdarzeń ("co się stało wczoraj o 14:03?") -- większość API
  nie przechowuje historii wywołań.

**Kafka rozwiązuje to, wprowadzając pośrednika**: aplikacja PUBLIKUJE zdarzenie
("zamówienie #123 złożone") RAZ, do Kafki -- nie wiedząc i nie dbając o to, KTO to
przeczyta. Każdy zainteresowany system SUBSKRYBUJE Kafkę i odczytuje zdarzenia NIEZALEŻNIE,
we własnym tempie. Dodanie nowego konsumenta NIE wymaga zmiany aplikacji źródłowej --
tylko podłączenia nowego czytelnika do Kafki.

> ⚡ **Kafka to rozproszony, uporządkowany DZIENNIK (log)**
>
> Najprostszy mentalny model Kafki: to gigantyczny, ciągle rosnący PLIK TEKSTOWY (log), do którego producenci DOPISUJĄ wiersze na końcu, a konsumenci CZYTAJĄ go od dowolnego miejsca, w swoim własnym tempie -- bez usuwania przeczytanych wierszy. Wiele niezależnych czytelników może czytać TEN SAM log jednocześnie, każdy w innym miejscu.

<a id="sec2"></a>
## 2. Podstawowe pojęcia: Topic, Partition, Producer, Consumer

- **Topic** (temat) -- nazwana kategoria zdarzeń, np. `zamowienia`, `kliknięcia_strony`,
  `platnosci`. Odpowiednik tabeli w bazie danych albo pliku w systemie plików.
- **Partition** (partycja) -- KAŻDY topic jest fizycznie podzielony na partycje (np. 6
  partycji dla topicu `zamowienia`) -- dokładnie ta sama idea co partycje DataFrame'u w
  Sparku (kurs Databricks, Moduł 2)! Partycje umożliwiają RÓWNOLEGŁE przetwarzanie: różne
  partycje mogą być czytane przez różnych konsumentów jednocześnie.
- **Producer** (producent) -- aplikacja, która PUBLIKUJE zdarzenia do topicu. Decyduje
  (albo zostawia to Kafce), do KTÓREJ partycji trafi każde zdarzenie -- zwykle na
  podstawie klucza (np. wszystkie zdarzenia dla tego samego `klient_id` trafiają zawsze
  do TEJ SAMEJ partycji, co gwarantuje ich kolejność).
- **Consumer** (konsument) -- aplikacja, która CZYTA zdarzenia z topicu, zaczynając od
  wybranego miejsca (offsetu, sekcja 3) i idąc naprzód w miarę pojawiania się nowych
  zdarzeń.
- **Broker** -- pojedynczy serwer Kafki; **klaster** Kafki to zbiór wielu brokerów
  współpracujących (dla skalowalności i odporności na awarie).

> 🏭 **Partycjonowanie po kluczu to decyzja o KOLEJNOŚCI**
>
> Kafka gwarantuje kolejność zdarzeń TYLKO WEWNĄTRZ jednej partycji, nie w całym topicu. Jeśli kolejność zdarzeń dla danego klienta jest ważna (np. "złożono zamówienie" musi być przetworzone PRZED "anulowano zamówienie"), producent MUSI użyć `klient_id` jako klucza partycjonowania, żeby wszystkie zdarzenia tego klienta trafiały zawsze do tej samej partycji.

<a id="sec3"></a>
## 3. Consumer Groups i offsety

**Offset** to numer porządkowy pozycji konsumenta W DANEJ PARTYCJI -- "przeczytałem już
zdarzenia 0-999, następne zacznę czytać od 1000". Kafka PRZECHOWUJE offsety konsumentów
(nie usuwa przeczytanych danych!) -- dzięki temu konsument może się zrestartować i
kontynuować DOKŁADNIE tam, gdzie skończył, zamiast czytać wszystko od nowa.

**Consumer Group** (grupa konsumentów) -- kilku konsumentów oznaczonych tą samą nazwą
grupy DZIELI między siebie partycje topicu -- każda partycja jest czytana przez DOKŁADNIE
JEDNEGO konsumenta w danej grupie w danym momencie (RÓWNOLEGŁE przetwarzanie danego
topicu). Różne grupy konsumentów są od siebie NIEZALEŻNE -- system analityczny i system
antyfraudowy mogą czytać TEN SAM topic `zamowienia`, każdy we własnym tempie, z własnym
zestawem offsetów, bez wzajemnego wpływu.

> ⚠️ **Więcej konsumentów niż partycji = konsumenci bezczynni**
>
> Jeśli topic ma 6 partycji, a Consumer Group ma 8 konsumentów, DWA z nich nigdy nie dostaną żadnej partycji do czytania -- będą bezczynne. Liczba partycji to GÓRNA GRANICA równoległości odczytu dla jednej grupy konsumentów -- to kluczowa decyzja projektowa przy tworzeniu topicu (podobnie jak liczba partycji Sparka wpływała na równoległość w kursie Databricks, Moduł 9).

<a id="sec4"></a>
## 4. Gwarancje dostarczenia

Trzy poziomy gwarancji, że zdarzenie zostanie PRZETWORZONE -- ważny wybór inżynierski,
bo mocniejsze gwarancje kosztują wydajność:

- **At-most-once** (najwyżej raz) -- zdarzenie może zostać ZGUBIONE, ale nigdy
  przetworzone dwukrotnie. Najszybsze, odpowiednie tam, gdzie utrata pojedynczego
  zdarzenia nie boli (np. metryki monitoringu, gdzie liczy się trend, nie pojedynczy
  punkt).
- **At-least-once** (co najmniej raz) -- zdarzenie NIGDY nie zostanie zgubione, ale może
  zostać przetworzone DWUKROTNIE (np. po restarcie konsumenta, który zdążył przetworzyć,
  ale nie zdążył zapisać nowego offsetu). Domyślne i najczęściej używane -- wymaga, żeby
  przetwarzanie było **idempotentne** (wielokrotne przetworzenie tego samego zdarzenia
  daje ten sam efekt, np. `UPSERT` zamiast `INSERT`).
- **Exactly-once** (dokładnie raz) -- najsilniejsza (i najdroższa) gwarancja: każde
  zdarzenie przetworzone DOKŁADNIE jeden raz. Kafka wspiera to przez tzw. *transactional
  producers* -- używane tam, gdzie duplikaty są niedopuszczalne (np. przetwarzanie
  płatności).

<a id="sec5"></a>
## 5. Kod producenta i konsumenta

Poniższy kod (biblioteka `kafka-python`) pokazuje, jak wyglądałby prosty producent i
konsument dla naszego znanego zestawu danych sklepu internetowego -- publikujący zdarzenia
"nowe zamówienie" w czasie rzeczywistym. Ten kod jest gotowy do uruchomienia na
prawdziwym brokerze (patrz ostrzeżenie na początku modułu), ale w tym środowisku
pozostaje przykładem do przeczytania, nie do wykonania.

```python
# PRODUCENT -- publikuje nowe zamowienia do topicu "zamowienia"
from kafka import KafkaProducer
import json

producent = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: str(k).encode("utf-8"),
)

nowe_zamowienie = {
    "zamowienie_id": 3001,
    "klient_id": 42,
    "wartosc": 259.99,
    "status": "Zlozone",
}

# klucz = klient_id -- gwarantuje, ze WSZYSTKIE zamowienia tego klienta
# traﬁaja do tej samej partycji (kolejnosc zachowana per klient)
producent.send("zamowienia", key=nowe_zamowienie["klient_id"], value=nowe_zamowienie)
producent.flush()
```

```python
# KONSUMENT -- czyta zamowienia z topicu "zamowienia" w ramach grupy "system_analityczny"
from kafka import KafkaConsumer
import json

konsument = KafkaConsumer(
    "zamowienia",
    bootstrap_servers="localhost:9092",
    group_id="system_analityczny",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    auto_offset_reset="earliest",  # przy pierwszym uruchomieniu: czytaj od poczatku topicu
)

for zdarzenie in konsument:
    zamowienie = zdarzenie.value
    print(f"Nowe zamowienie #{zamowienie['zamowienie_id']} od klienta {zamowienie['klient_id']}")
    # tutaj: logika biznesowa -- np. zapis do hurtowni, aktualizacja dashboardu na zywo
```

> ⚡ **`auto_offset_reset` decyduje, OD KTÓREGO miejsca zacząć**
>
> `"earliest"` czyta CAŁĄ dostępną historię topicu od początku (dobre przy pierwszym uruchomieniu nowego konsumenta). `"latest"` (domyślne) czyta TYLKO nowe zdarzenia od momentu podłączenia konsumenta -- historia sprzed tego momentu jest pomijana. Wybór zależy od tego, czy nowy konsument potrzebuje pełnej historii, czy tylko danych "od teraz".

<a id="sec6"></a>
## 6. Kafka Connect i Schema Registry

Dwa narzędzia z ekosystemu Kafki, warte znajomości nawet bez uruchamiania ich samodzielnie:

- **Kafka Connect** -- framework do łączenia Kafki z zewnętrznymi systemami BEZ pisania
  kodu producenta/konsumenta ręcznie -- gotowe "złącza" (*connectors*) potrafią np.
  automatycznie publikować do Kafki KAŻDĄ zmianę w tabeli PostgreSQL (technika zwana
  **CDC** -- *Change Data Capture*, częsty sposób na strumieniowe zasilanie hurtowni
  danymi z baz transakcyjnych) albo automatycznie zapisywać dane z Kafki do S3/hurtowni.
- **Schema Registry** -- centralne repozytorium SCHEMATÓW wiadomości (np. w formacie
  Avro/Protobuf) -- gwarantuje, że producent i konsument ZGADZAJĄ SIĘ co do struktury
  danych, i pozwala bezpiecznie EWOLUOWAĆ schemat w czasie (dodanie nowej kolumny nie
  psuje starych konsumentów) -- podobny problem do ewolucji schematu w Delta Lake
  (kurs Databricks, Moduł 10).

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy (koncepcyjnie):
- dlaczego Kafka istnieje -- rozprzężenie (decoupling) producentów od konsumentów,
- topic, partition, producer, consumer, broker/klaster,
- consumer groups i offsety -- jak Kafka umożliwia RÓWNOLEGŁE, NIEZALEŻNE czytanie tego
  samego topicu przez wiele systemów,
- trzy gwarancje dostarczenia: at-most-once, at-least-once, exactly-once,
- podstawowy kod producenta i konsumenta (`kafka-python`),
- Kafka Connect (CDC z baz danych) i Schema Registry (ewolucja schematu).

Poniższe ćwiczenia są PROJEKTOWE/koncepcyjne -- nie wymagają uruchomienia Kafki, tylko
przemyślenia architektury.

### 📝 Ćwiczenie 1: Ile partycji?

Nasz sklep internetowy generuje ok. 3000 zamówień DZIENNIE, a przewidywany wzrost to 10x w ciągu 2 lat. Zaproponuj liczbę partycji dla topicu `zamowienia` i uzasadnij -- pamiętaj, że dodanie partycji PÓŹNIEJ jest możliwe, ale zmienia przypisanie kluczy do partycji (może zaburzyć kolejność zdarzeń dla istniejących kluczy).

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# Sensowna odpowiedz: zaczac od WIECEJ partycji niz obecnie potrzeba (np. 12),
# zeby miec zapas na wzrost 10x BEZ koniecznosci zwiekszania liczby partycji pozniej
# (co zaburzyloby przypisanie klient_id -> partycja dla istniejacych danych).
# Zbyt duzo partycji na starcie (np. 500) tez jest zle -- kazda partycja to dodatkowy
# narzut (pliki, metadane, polaczenia) -- 12-24 partycje to rozsadny kompromis dla
# takiej skali.
print("Zobacz uzasadnienie w komentarzu powyzej.")
```

</details>

### 📝 Ćwiczenie 2: At-least-once czy exactly-once?

Dla każdego przypadku zdecyduj, jaka gwarancja dostarczenia jest NAJBARDZIEJ odpowiednia i uzasadnij: (a) publikowanie metryk zużycia CPU serwerów co sekundę, (b) przetwarzanie płatności kartą, (c) wysyłanie e-maili powitalnych do nowych klientów.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# (a) at-most-once wystarczy (a nawet lepszy pod wzgledem kosztu) -- utrata
#     pojedynczego pomiaru CPU nie ma znaczenia przy tysiacach pomiarow na godzine.
# (b) exactly-once (albo at-least-once + idempotentne przetwarzanie, np. sprawdzenie
#     "czy ta platnosc juz zostala rozliczona" przed ponownym obciazeniem karty) --
#     podwojne obciazenie karty klienta jest niedopuszczalne.
# (c) at-least-once z idempotencja (sprawdz, czy e-mail juz zostal wyslany, zanim
#     wyslesz ponownie) -- duplikat e-maila to niewygoda, nie katastrofa, ale
#     lepiej tego unikac.
print("Zobacz komentarze powyzej.")
```

</details>

### 📝 Ćwiczenie 3: Zaprojektuj klucz partycjonowania

Dla topicu `kliknięcia_strony` (każde zdarzenie to jedno kliknięcie użytkownika na stronie sklepu) zaproponuj klucz partycjonowania i uzasadnij, jaką WŁAŚCIWOŚĆ kolejności chcesz zagwarantować.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# Klucz: id_sesji_uzytkownika (nie klient_id!) -- gwarantuje, ze WSZYSTKIE klikniecia
# W RAMACH JEDNEJ SESJI traﬁaja do tej samej partycji, wiec konsument widzi je we
# WLASCIWEJ KOLEJNOSCI (np. do rekonstrukcji sciezki uzytkownika po stronie: "wszedl
# na strone glowna" -> "kliknal produkt" -> "dodal do koszyka"). Uzycie klient_id
# byloby gorsze dla anonimowych uzytkownikow (brak klient_id przed zalogowaniem).
print("Zobacz uzasadnienie w komentarzu powyzej.")
```

</details>

> 🔥 **Wyzwanie: zaprojektuj architekturę CDC dla naszego sklepu**
>
> Nasz sklep ma bazę PostgreSQL (jak w kursie SQL) z tabelami `klienci`, `zamowienia`, `pozycje_zamowien`. Zaprojektuj (opisz jednym akapitem, bez kodu) architekturę, która automatycznie publikuje KAŻDĄ zmianę w tych tabelach do osobnych topiców Kafki w czasie rzeczywistym, skąd Spark Structured Streaming (Moduł 9) buduje zawsze aktualną kopię danych w lakehouse. Jakie narzędzie z sekcji 6 tu pasuje? Jakie topiki byś stworzył?

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# Architektura: Kafka Connect z konektorem CDC (np. Debezium) podlaczonym do
# PostgreSQL -- odczytuje write-ahead log (WAL) bazy i publikuje KAZDA zmiane
# (INSERT/UPDATE/DELETE) jako zdarzenie do Kafki, bez modyfikowania kodu aplikacji.
#
# Topiki: jeden per tabela zrodlowa -- "cdc.klienci", "cdc.zamowienia",
# "cdc.pozycje_zamowien" -- kazde zdarzenie zawiera stan PRZED i PO zmianie oraz typ
# operacji.
#
# Spark Structured Streaming (Modul 9) subskrybuje te topiki i na biezaco aktualizuje
# tabele w lakehouse (warstwa bronze/silver z Modulu 11 kursu Databricks) -- baza
# PostgreSQL pozostaje jedynym "zrodlem prawdy", a hurtownia analityczna jest zawsze
# swieza, bez kosztownych, okresowych pelnych wyciagow (full extract) z bazy produkcyjnej.
print("Zobacz opis architektury w komentarzu powyzej.")
```

To dokladnie realny wzorzec produkcyjny (Debezium + Kafka + Spark Structured Streaming) uzywany przez wiele firm do zasilania lakehouse'ow danymi z baz transakcyjnych bez obciazania ich okresowymi, ciezkimi zapytaniami.

</details>

## Co dalej?

W **Module 9** przechodzimy od TEORII streamingu do PRAKTYKI: uruchamiamy **Spark
Structured Streaming** naprawdę, lokalnie -- używając wbudowanego źródła `rate` jako
bezpiecznego zamiennika prawdziwego strumienia Kafki, żeby zobaczyć okna czasowe,
watermarking i tryby wyjścia w akcji.